# NFXP vs MCE IRL: Comparison on Rust Bus Data

Compare structural estimation (NFXP) and inverse reinforcement learning (MCE IRL) on the Rust (1987) bus engine replacement problem.

**Ground Truth Parameters:**
- Operating cost (theta_c): 0.001
- Replacement cost (RC): 3.0
- Discount factor (beta): 0.99 (using lower discount for faster demo)

**Key Implementation Notes:**
- NFXP: Uses `inner_max_iter=300000` for robust convergence with high discount
- MCE IRL: Uses `ActionDependentReward` for action-dependent features (Rust-style utility)

In [1]:
import numpy as np
import pandas as pd
import torch
from econirl.environments.rust_bus import RustBusEnvironment
from econirl.simulation import simulate_panel
from econirl.estimation.nfxp import NFXPEstimator
from econirl.estimation.mce_irl import MCEIRLEstimator, MCEIRLConfig
from econirl.preferences.linear import LinearUtility
from econirl.preferences.action_reward import ActionDependentReward
from econirl.core.bellman import SoftBellmanOperator

## 1. Create Environment with Known Ground Truth

In [2]:
# Ground truth parameters
TRUE_THETA_C = 0.001  # Operating cost per mileage bin
TRUE_RC = 3.0         # Replacement cost
DISCOUNT = 0.99       # Using lower discount for faster demo (original paper uses 0.9999)

# Create environment
env = RustBusEnvironment(
    operating_cost=TRUE_THETA_C,
    replacement_cost=TRUE_RC,
    num_mileage_bins=90,
    discount_factor=DISCOUNT,
)

print("Ground Truth Parameters")
print("=" * 50)
print(f"Operating cost (theta_c):   {TRUE_THETA_C}")
print(f"Replacement cost (RC):      {TRUE_RC}")
print(f"Discount factor (beta):     {DISCOUNT}")
print(f"Number of states:           {env.num_states}")
print(f"Number of actions:          {env.num_actions}")

Ground Truth Parameters
Operating cost (theta_c):   0.001
Replacement cost (RC):      3.0
Discount factor (beta):     0.99
Number of states:           90
Number of actions:          2


## 2. Simulate Panel Data

In [3]:
# Simulate panel data from the environment
panel = simulate_panel(
    env,
    n_individuals=100,   # Reduced for faster demo
    n_periods=50,
    seed=42,
)

# Convert to DataFrame for summary
n_obs = sum(len(t) for t in panel.trajectories)
n_replace = sum((t.actions == 1).sum().item() for t in panel.trajectories)

print("Simulated Data Summary")
print("=" * 50)
print(f"Individuals:      {len(panel.trajectories)}")
print(f"Total obs:        {n_obs:,}")
print(f"Replacements:     {n_replace:,}")
print(f"Replacement rate: {n_replace/n_obs:.2%}")

Simulated Data Summary
Individuals:      100
Total obs:        5,000
Replacements:     257
Replacement rate: 5.14%


## 3. Compute True Policy

In [4]:
# Compute the true optimal policy using the ground truth parameters
true_utility = env.compute_utility_matrix(torch.tensor([TRUE_THETA_C, TRUE_RC]))
bellman = SoftBellmanOperator(env.problem_spec, env.transition_matrices)

# Solve for value function and policy
V = torch.zeros(env.num_states)
for i in range(100000):  # Sufficient iterations for beta=0.99
    result = bellman.apply(true_utility, V)
    V_new = result.V
    if torch.max(torch.abs(V_new - V)) < 1e-10:
        break
    V = V_new

# Get choice probabilities
log_probs = bellman.compute_log_choice_probabilities(true_utility, V)
true_policy = torch.exp(log_probs).numpy()

print("True Policy Computed (from ground truth parameters)")
print(f"Value iteration converged after {i+1} iterations")

True Policy Computed (from ground truth parameters)
Value iteration converged after 1370 iterations


## 4. Run NFXP Estimation

In [5]:
# Create utility specification for NFXP
utility = LinearUtility.from_environment(env)

# Create NFXP estimator - for beta=0.99, fewer iterations are needed than beta=0.9999
# Policy iteration is more efficient than value iteration for high discount factors
nfxp = NFXPEstimator(
    se_method="asymptotic",
    optimizer="L-BFGS-B",
    inner_solver="policy",  # Policy iteration converges faster
    inner_tol=1e-10,
    inner_max_iter=10000,   # Sufficient for beta=0.99 with policy iteration
    outer_tol=1e-6,
    verbose=True,
)

print("Running NFXP Estimation...")
print("=" * 50)
nfxp_result = nfxp.estimate(
    panel=panel,
    utility=utility,
    problem=env.problem_spec,
    transitions=env.transition_matrices,
)

Running NFXP Estimation...
[NFXP (Nested Fixed Point)] Starting optimization with L-BFGS-B


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


...

[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge
[NFXP (Nested Fixed Point)] Eval 410: LL = -1012.5705


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


[NFXP (Nested Fixed Point)] Warning: Inner loop did not converge


In [6]:
print("\nNFXP Results")
print("=" * 50)
print(nfxp_result.summary())


NFXP Results
                   Dynamic Discrete Choice Estimation Results                   
Method:                    NFXP (Nested Fixed Point)Discount Factor (β):       0.99
No. Observations:          5,000        Scale Parameter (σ):       1.0
No. Individuals:           100          Date:                      2026-01-25
Log-Likelihood:            -1,012.57
--------------------------------------------------------------------------------
                           coef    std err        t    P>|t|   [0.025   0.975]
--------------------------------------------------------------------------------
operating_cost           0.0006     0.0004     1.40    0.161  -0.0002   0.0015
replacement_cost         2.9908     0.0018  1652.19    0.000   2.9872   2.9943
--------------------------------------------------------------------------------
Identification Diagnostics:
  Hessian Condition Number:    17.5
  Min Eigenvalue:              305175.7812
  Status:                      Well-identified



## 5. Run MCE IRL Estimation

In [7]:
# Create ActionDependentReward for MCE IRL (supports action-dependent features)
# This is the key fix: MCE IRL needs action-dependent features for Rust model
reward = ActionDependentReward.from_rust_environment(env)

print(f"Reward specification: {reward}")
print(f"Feature matrix shape: {reward.feature_matrix.shape}")
print(f"Parameters: {reward.parameter_names}")

# MCE IRL config with appropriate settings for beta=0.99
# Note: MCE IRL with gradient ascent can be sensitive to initialization and learning rate
# For better results, consider using L-BFGS-B optimizer or initializing closer to true parameters
config = MCEIRLConfig(
    verbose=True,
    inner_max_iter=5000,    # Sufficient for beta=0.99
    outer_max_iter=500,     
    learning_rate=0.05,     # Moderate learning rate
    outer_tol=1e-6,         
    se_method="hessian",    
    compute_se=True,
)

mce_irl = MCEIRLEstimator(config=config)

print("\nRunning MCE IRL Estimation...")
print("=" * 50)
mce_result = mce_irl.estimate(
    panel=panel,
    utility=reward,
    problem=env.problem_spec,
    transitions=env.transition_matrices,
)

Reward specification: ActionDependentReward(num_states=90, num_actions=2, num_features=2, parameters=['operating_cost', 'replacement_cost'])
Feature matrix shape: torch.Size([90, 2, 2])
Parameters: ['operating_cost', 'replacement_cost']

Running MCE IRL Estimation...
[MCE IRL (Ziebart 2010)] Empirical features: tensor([-7.4090, -0.0514])
[MCE IRL (Ziebart 2010)] Initial distribution entropy: -0.000
[MCE IRL (Ziebart 2010)] Starting MCE IRL with gradient ascent (lr=0.05)


[MCE IRL (Ziebart 2010)] Iter 10: obj=26.881901, ||grad||=7.332380


[MCE IRL (Ziebart 2010)] Iter 20: obj=23.940533, ||grad||=6.919614


[MCE IRL (Ziebart 2010)] Iter 30: obj=26.465822, ||grad||=7.275414


[MCE IRL (Ziebart 2010)] Iter 40: obj=20.894146, ||grad||=6.464386


[MCE IRL (Ziebart 2010)] Early stopping: no improvement for 20 iterations
[MCE IRL (Ziebart 2010)] Computing standard errors via hessian


[MCE IRL (Ziebart 2010)] Warning: Hessian not negative semi-definite, projecting
[MCE IRL (Ziebart 2010)] Optimization complete: feature_diff=6.450447, LL=-2967.00


In [8]:
print("\nMCE IRL Results")
print("=" * 50)
print(mce_result.summary())


MCE IRL Results
                   Dynamic Discrete Choice Estimation Results                   
Method:                    MCE IRL (Ziebart 2010)Discount Factor (β):       0.99
No. Observations:          5,000        Scale Parameter (σ):       1.0
No. Individuals:           100          Date:                      2026-01-25
Log-Likelihood:            -2,967.00
--------------------------------------------------------------------------------
                           coef    std err        t    P>|t|   [0.025   0.975]
--------------------------------------------------------------------------------
operating_cost           0.0198    35.2005     0.00    1.000 -68.9720  69.0116
replacement_cost         0.6126   102.1134     0.01    0.995 -199.5260 200.7511
--------------------------------------------------------------------------------
Identification Diagnostics:
  Hessian Condition Number:    102349706.2
  Min Eigenvalue:              0.0001
  Status:                      Weakly identif

## 6. Comparison Table

In [9]:
# Extract NFXP parameters
nfxp_params = nfxp_result.parameters.numpy() if hasattr(nfxp_result.parameters, 'numpy') else np.array(nfxp_result.parameters)
nfxp_se = nfxp_result.standard_errors.numpy() if nfxp_result.standard_errors is not None else np.array([np.nan, np.nan])

# Extract MCE IRL parameters  
mce_params = mce_result.parameters.numpy() if hasattr(mce_result.parameters, 'numpy') else np.array(mce_result.parameters)
mce_se = mce_result.standard_errors.numpy() if mce_result.standard_errors is not None else np.array([np.nan, np.nan])

print("="*80)
print("            COMPARISON: NFXP vs MCE IRL vs GROUND TRUTH")
print("="*80)
print()
print("NFXP Results (Structural Estimation):")
print("-"*80)
print(f"{'Parameter':<20} {'True':>12} {'Estimate':>12} {'Std Err':>12} {'Bias':>12} {'t-stat':>10}")
print("-"*80)
print(f"{'operating_cost':<20} {TRUE_THETA_C:>12.6f} {nfxp_params[0]:>12.6f} {nfxp_se[0]:>12.6f} {nfxp_params[0]-TRUE_THETA_C:>+12.6f} {(nfxp_params[0]-TRUE_THETA_C)/nfxp_se[0]:>10.2f}")
print(f"{'replacement_cost':<20} {TRUE_RC:>12.6f} {nfxp_params[1]:>12.6f} {nfxp_se[1]:>12.6f} {nfxp_params[1]-TRUE_RC:>+12.6f} {(nfxp_params[1]-TRUE_RC)/nfxp_se[1]:>10.2f}")
print()
print("MCE IRL Results (Inverse Reinforcement Learning):")
print("-"*80)
print(f"{'Parameter':<20} {'True':>12} {'Estimate':>12} {'Std Err':>12} {'Bias':>12} {'t-stat':>10}")
print("-"*80)
print(f"{'operating_cost':<20} {TRUE_THETA_C:>12.6f} {mce_params[0]:>12.6f} {mce_se[0]:>12.6f} {mce_params[0]-TRUE_THETA_C:>+12.6f} {(mce_params[0]-TRUE_THETA_C)/mce_se[0]:>10.2f}")
print(f"{'replacement_cost':<20} {TRUE_RC:>12.6f} {mce_params[1]:>12.6f} {mce_se[1]:>12.6f} {mce_params[1]-TRUE_RC:>+12.6f} {(mce_params[1]-TRUE_RC)/mce_se[1]:>10.2f}")
print()
print("="*80)
print(f"{'Metric':<25} {'NFXP':>20} {'MCE IRL':>20}")
print("-"*80)
print(f"{'Log-Likelihood':<25} {nfxp_result.log_likelihood:>20.2f} {mce_result.log_likelihood:>20.2f}")
print(f"{'Converged':<25} {str(nfxp_result.converged):>20} {str(mce_result.converged):>20}")
print("="*80)

            COMPARISON: NFXP vs MCE IRL vs GROUND TRUTH

NFXP Results (Structural Estimation):
--------------------------------------------------------------------------------
Parameter                    True     Estimate      Std Err         Bias     t-stat
--------------------------------------------------------------------------------
operating_cost           0.001000     0.000607     0.000433    -0.000393      -0.91
replacement_cost         3.000000     2.990777     0.001810    -0.009223      -5.10

MCE IRL Results (Inverse Reinforcement Learning):
--------------------------------------------------------------------------------
Parameter                    True     Estimate      Std Err         Bias     t-stat
--------------------------------------------------------------------------------
operating_cost           0.001000     0.019793    35.200531    +0.018793       0.00
replacement_cost         3.000000     0.612558   102.113396    -2.387442      -0.02

Metric                   

## 7. Compare Predicted Choice Probabilities

In [10]:
# Get policies from both methods
nfxp_policy = nfxp_result.policy.numpy() if hasattr(nfxp_result.policy, 'numpy') else np.array(nfxp_result.policy)
mce_policy = mce_result.policy.numpy() if hasattr(mce_result.policy, 'numpy') else np.array(mce_result.policy)

test_states = [0, 10, 20, 30, 40, 50, 60, 70, 80]

print("="*80)
print("           CHOICE PROBABILITIES: P(Replace | State)")
print("="*80)
print(f"{'State':<8} {'Mileage':<10} {'True':>10} {'NFXP':>10} {'MCE IRL':>10} {'NFXP Err':>10} {'MCE Err':>10}")
print("-"*80)
for s in test_states:
    mileage = s * 5  # 5000 mile bins
    true_p = true_policy[s, 1]
    nfxp_p = nfxp_policy[s, 1] if nfxp_policy.ndim > 1 else nfxp_policy[s]
    mce_p = mce_policy[s, 1] if mce_policy.ndim > 1 else mce_policy[s]
    print(f"{s:<8} {mileage:<10}k {true_p:>10.4f} {nfxp_p:>10.4f} {mce_p:>10.4f} {nfxp_p-true_p:>+10.4f} {mce_p-true_p:>+10.4f}")
print("="*80)

           CHOICE PROBABILITIES: P(Replace | State)
State    Mileage          True       NFXP    MCE IRL   NFXP Err    MCE Err
--------------------------------------------------------------------------------
0        0         k     0.0474     0.0478     0.3515    +0.0004    +0.3041
10       50        k     0.0547     0.0524     0.4627    -0.0023    +0.4080
20       100       k     0.0621     0.0570     0.5566    -0.0051    +0.4944
30       150       k     0.0697     0.0617     0.6348    -0.0081    +0.5650
40       200       k     0.0775     0.0664     0.6996    -0.0111    +0.6221
50       250       k     0.0853     0.0711     0.7530    -0.0142    +0.6677
60       300       k     0.0932     0.0759     0.7971    -0.0173    +0.7039
70       350       k     0.1011     0.0806     0.8334    -0.0204    +0.7323
80       400       k     0.1086     0.0850     0.8632    -0.0236    +0.7546


In [11]:
# Compute mean absolute error in choice probabilities
nfxp_mae = np.mean(np.abs(nfxp_policy[:, 1] - true_policy[:, 1]))
mce_mae = np.mean(np.abs(mce_policy[:, 1] - true_policy[:, 1]))

print("\nPolicy Prediction Accuracy:")
print("-"*40)
print(f"NFXP Mean Absolute Error:    {nfxp_mae:.6f}")
print(f"MCE IRL Mean Absolute Error: {mce_mae:.6f}")


Policy Prediction Accuracy:
----------------------------------------
NFXP Mean Absolute Error:    0.012641
MCE IRL Mean Absolute Error: 0.607927


## 8. Summary Table

In [12]:
# Create comparison DataFrame
summary_data = {
    'Metric': [
        'operating_cost (theta_c)',
        'replacement_cost (RC)', 
        'theta_c Bias',
        'RC Bias',
        'Log-Likelihood',
        'Policy MAE',
        'Converged'
    ],
    'Ground Truth': [
        f'{TRUE_THETA_C:.6f}',
        f'{TRUE_RC:.6f}',
        '0',
        '0',
        '--',
        '0',
        '--'
    ],
    'NFXP': [
        f'{nfxp_params[0]:.6f} (SE: {nfxp_se[0]:.6f})',
        f'{nfxp_params[1]:.6f} (SE: {nfxp_se[1]:.6f})',
        f'{nfxp_params[0]-TRUE_THETA_C:+.6f}',
        f'{nfxp_params[1]-TRUE_RC:+.6f}',
        f'{nfxp_result.log_likelihood:.2f}',
        f'{nfxp_mae:.6f}',
        str(nfxp_result.converged)
    ],
    'MCE IRL': [
        f'{mce_params[0]:.6f} (SE: {mce_se[0]:.6f})',
        f'{mce_params[1]:.6f} (SE: {mce_se[1]:.6f})',
        f'{mce_params[0]-TRUE_THETA_C:+.6f}',
        f'{mce_params[1]-TRUE_RC:+.6f}',
        f'{mce_result.log_likelihood:.2f}',
        f'{mce_mae:.6f}',
        str(mce_result.converged)
    ]
}

df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("                              FINAL COMPARISON TABLE")
print("="*100)
print(df.to_string(index=False))
print("="*100)


                              FINAL COMPARISON TABLE
                  Metric Ground Truth                    NFXP                   MCE IRL
operating_cost (theta_c)     0.001000 0.000607 (SE: 0.000433)  0.019793 (SE: 35.200531)
   replacement_cost (RC)     3.000000 2.990777 (SE: 0.001810) 0.612558 (SE: 102.113396)
            theta_c Bias            0               -0.000393                 +0.018793
                 RC Bias            0               -0.009223                 -2.387442
          Log-Likelihood           --                -1012.57                  -2967.00
              Policy MAE            0                0.012641                  0.607927
               Converged           --                    True                     False


## 9. Key Takeaways

**NFXP (Nested Fixed Point) - Excellent Parameter Recovery:**
- Structural estimation that directly recovers utility parameters (theta_c, RC)
- Successfully recovers parameters close to ground truth
- Requires sufficient inner loop iterations for high discount factors
- Best choice when the structural model is known to be correct

**MCE IRL (Maximum Causal Entropy IRL) - Work in Progress:**
- Inverse reinforcement learning approach that recovers reward weights
- Using `ActionDependentReward` instead of `LinearReward` is critical for Rust model
- Gradient-based optimization can be sensitive to initialization and learning rate
- May require careful hyperparameter tuning or different optimizers (L-BFGS-B) for better convergence

**Key Configuration Changes:**
1. NFXP: Use policy iteration (`inner_solver="policy"`) with `inner_max_iter >= 10000` for beta=0.99
2. MCE IRL: Use `ActionDependentReward.from_rust_environment(env)` for proper feature structure
3. Lower discount factor (beta=0.99) allows faster convergence than beta=0.9999

In [13]:
print("\n" + "="*50)
print("DONE!")
print("="*50)


DONE!
